# 밀폐 공간 전술 정찰 및 구조를 위한 온디바이스 임베디드 AI 드론
## 3-OFF + MAP: GPS-Denied 실내 객체 탐지 모델 학습

이 노트북은 **Google Colab** 환경에서 실행되도록 최적화되어 있습니다.
런타임 유형을 반드시 **GPU**로 설정하고 진행해 주세요.

In [1]:
# 1. 환경 설정 및 GPU 확인
!pip install ultralytics
from ultralytics import YOLO
import torch
import os

print(f"GPU 확인: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"현재 사용 중인 GPU: {torch.cuda.get_device_name(0)}")

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 9.3 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 3.4 MB/s  0:00:00
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   --- ------------------------------------ 3.7/44.0 MB 19.8 MB/s eta 0:00:03
   -------- ------------------------------- 9.7/44.0 MB 24.0 MB/s eta 0:00:02
   -------------- ------------------------- 16.0/44.0 MB 25.9 MB/s eta 0:00:02
   --------------------- ------------------ 23.3/44.0 MB 28.4 MB/s eta 0:00:01
   ----------------------------- ---------- 32.5/44.0 MB 31.9 MB/s eta 0:00:01
   ---------------------------------------  43.3/44.0 MB 34.9 MB/s eta 0:00:01
   ---------------------------------------- 44.0/44.0 MB 30.4 MB/s  0:00:01
   ---------------------------------------- 0.0/837.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/837.6 kB ? eta -:--:--
   -------------------

## 2. 대용량 데이터셋 고속 다운로드
Colab의 초고속 인터넷망을 활용하여 1GB 이상의 데이터셋을 빠르게 다운로드합니다.

In [2]:
# COCO val2017 (약 800MB) 이미지 및 어노테이션 다운로드
os.makedirs('/content/datasets/coco', exist_ok=True)

print("COCO 데이터셋 다운로드 중...")
!wget -q -nc http://images.cocodataset.org/zips/val2017.zip -P /content/datasets/coco
!wget -q -nc http://images.cocodataset.org/annotations/annotations_trainval2017.zip -P /content/datasets/coco

print("압축 해제 중...")
!unzip -q /content/datasets/coco/val2017.zip -d /content/datasets/coco
!unzip -q /content/datasets/coco/annotations_trainval2017.zip -d /content/datasets/coco
print("COCO 데이터셋 준비 완료!")

COCO 데이터셋 다운로드 중...


'wget'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


압축 해제 중...


'wget'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.
'unzip'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


COCO 데이터셋 준비 완료!


'unzip'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [3]:
# D-Fire 데이터셋 다운로드 (Kaggle)
# 주의: 이 셀을 실행하기 전에 Kaggle API 키(kaggle.json)를 Colab 환경에 업로드해야 합니다.
print("D-Fire 데이터셋 다운로드는 Kaggle API 설정이 필요합니다.")
print("kaggle.json 파일을 업로드하신 후 아래 코드를 주석 해제하여 실행하세요.")

# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d dataclusterlabs/fire-and-smoke-dataset -p /content/datasets/dfire --unzip

D-Fire 데이터셋 다운로드는 Kaggle API 설정이 필요합니다.
kaggle.json 파일을 업로드하신 후 아래 코드를 주석 해제하여 실행하세요.


## 3. 모델 학습
다운로드한 데이터를 기반으로 YOLOv8n 모델 전이학습을 시작합니다.

In [4]:
model = YOLO('yolov8n.pt')
results = model.train(
    data='dataset.yaml', # 실제 데이터 경로가 포함된 yaml 파일로 수정 필요
    epochs=50, 
    imgsz=640, 
    batch=16, 
    optimizer='AdamW', 
    lr0=0.001, 
    patience=10, 
    device=0, 
    seed=42
)

Ultralytics 8.4.102  Python-3.13.9 torch-2.13.0+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


In [5]:
print("모델 평가 완료. mAP50, mAP50-95, Precision, Recall 등의 지표를 확인하세요.")
# Jetson 보드 배포를 위한 ONNX 및 TensorRT 준비 변환
model.export(format='onnx')

모델 평가 완료. mAP50, mAP50-95, Precision, Recall 등의 지표를 확인하세요.
Ultralytics 8.4.102  Python-3.13.9 torch-2.13.0+cpu CPU (12th Gen Intel Core i3-1215U)
 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.9 environment at: c:\Anaconda3
Resolved 12 packages in 636ms
 Downloaded onnxruntime
 Downloaded onnx
Prepared 5 packages in 17.09s
Installed 5 packages in 6.37s
 + flatbuffers==25.12.19
 + ml-dtypes==0.5.4
 + onnx==1.22.0
 + onnxruntime==1.27.0
 + onnxslim==0.1.94

requirements: AutoUpdate success  26.5s
WARNING requirements: Restart runtime or rerun comman

'yolov8n.onnx'